In [1]:
# ============================================================
#  TITANIC — COMPLETE CODE  |  Days 2 to 5
#  Kaggle ML 30-Day Roadmap
#  Covers: EDA → Cleaning → Feature Engineering → Model
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# ============================================================
# DAY 2 — Load Data & EDA (Exploratory Data Analysis)
# ============================================================

train = pd.read_csv('/kaggle/input/datasets/faisalimam19/titanicdata/train.csv')
test  = pd.read_csv('/kaggle/input/datasets/faisalimam19/titanicdata/test.csv')

# --- Basic overview ---
print("=" * 50)
print("TRAIN SHAPE:", train.shape)
print("TEST SHAPE :", test.shape)

print("\n--- First 5 rows ---")
print(train.head())

print("\n--- Data Types & Non-Null Counts ---")
print(train.info())

print("\n--- Summary Statistics ---")
print(train.describe())

# --- Missing values ---
print("\n--- Missing Values (%) ---")
missing = (train.isnull().sum() / len(train) * 100).sort_values(ascending=False)
print(missing[missing > 0])

# --- Survival rate ---
print("\n--- Overall Survival Rate ---")
print(train['Survived'].value_counts(normalize=True).round(2))

# --- Survival by key features ---
print("\n--- Survival by Sex ---")
print(train.groupby('Sex')['Survived'].mean().round(2))

print("\n--- Survival by Pclass ---")
print(train.groupby('Pclass')['Survived'].mean().round(2))

print("\n--- Survival by Sex + Pclass ---")
print(train.groupby(['Sex', 'Pclass'])['Survived'].mean().round(2))


# ============================================================
# DAY 3 — Data Cleaning
# ============================================================

def clean_data(df):
    df = df.copy()

    # 1. Fill Age with median
    df['Age'] = df['Age'].fillna(df['Age'].median())

    # 2. Fill Embarked with mode (most common)
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

    # 3. Fill Fare with median (test set has 1 missing)
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())

    # 4. Cabin — too many missing, create binary flag instead
    df['HasCabin'] = df['Cabin'].notnull().astype(int)

    # 5. Drop columns we won't use
    df.drop(['Cabin', 'Ticket', 'PassengerId'], axis=1, inplace=True)

    return df

train = clean_data(train)
test  = clean_data(test)

print("\n--- After Cleaning — Missing Values ---")
print(train.isnull().sum())


# ============================================================
# DAY 4 — Feature Engineering
# ============================================================

def engineer_features(df):
    df = df.copy()

    # 1. Title — extracted from Name
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

    # Group rare titles
    df['Title'] = df['Title'].replace(
        ['Lady', 'Countess', 'Capt', 'Col', 'Don',
         'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
    )
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

    # 2. FamilySize & IsAlone
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']    = (df['FamilySize'] == 1).astype(int)

    # 3. FareBand — bin Fare into 4 buckets
    df['FareBand'] = pd.qcut(df['Fare'], 4, labels=[0, 1, 2, 3]).astype(int)

    # 4. AgeBand — bin Age into life stages
    df['AgeBand'] = pd.cut(
        df['Age'],
        bins=[0, 12, 18, 35, 60, 100],
        labels=[0, 1, 2, 3, 4]
    ).astype(int)

    # 5. Sex × Pclass interaction feature (your Day 4 insight!)
    df['Sex_Class'] = df['Sex'].astype(str) + '_' + df['Pclass'].astype(str)

    # 6. Encode Sex
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

    # 7. One-hot encode categorical columns
    df = pd.get_dummies(df, columns=['Title', 'Embarked', 'Sex_Class'], drop_first=True)

    # 8. Drop Name (no longer needed after extracting Title)
    df.drop('Name', axis=1, inplace=True)

    return df

train = engineer_features(train)
test  = engineer_features(test)

# Align test columns to match train (handles any missing dummies)
test = test.reindex(columns=train.drop('Survived', axis=1).columns, fill_value=0)

print("\n--- After Feature Engineering ---")
print(train.dtypes)
print("\nFeatures:", train.columns.tolist())


# ============================================================
# DAY 5 — Logistic Regression Model
# ============================================================

# --- Step 1: Prepare X and y ---
X = train.drop('Survived', axis=1)
y = train['Survived']

print("\n--- X shape:", X.shape)
print("--- y shape:", y.shape)

# --- Step 2: Train / Validation Split ---
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # keep same survival ratio in both halves
)

print(f"\nTraining samples  : {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

# --- Step 3: Feature Scaling ---
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # learn scale + apply
X_val_scaled   = scaler.transform(X_val)          # only apply — never relearn

# --- Step 4: Train Model ---
model = LogisticRegression(random_state=42, max_iter=200)
model.fit(X_train_scaled, y_train)
print("\nModel trained ✅")

# --- Step 5: Evaluate ---
y_pred = model.predict(X_val_scaled)

print(f"\nValidation Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print("\n=== Classification Report ===")
print(classification_report(y_val, y_pred, target_names=['Died', 'Survived']))

# --- Step 6: What did the model learn? ---
feature_weights = pd.DataFrame({
    'Feature': X.columns,
    'Weight' : model.coef_[0].round(3)
}).sort_values('Weight', ascending=False)

print("\n=== Feature Weights (what model learned) ===")
print(feature_weights.to_string(index=False))

# --- Step 7: Generate Kaggle Submission ---
X_test_scaled = scaler.transform(test)   # same scaler — never refit on test

test_preds = model.predict(X_test_scaled)

submission = pd.DataFrame({
    'PassengerId': pd.read_csv('/kaggle/input/datasets/faisalimam19/titanicdata/test.csv')['PassengerId'],
    'Survived'   : test_preds
})

submission.to_csv('submission.csv', index=False)
print("\n--- Submission Value Counts ---")
print(submission['Survived'].value_counts())
print("\nsubmission.csv saved ✅")

TRAIN SHAPE: (891, 12)
TEST SHAPE : (418, 11)

--- First 5 rows ---
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C12

In [2]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np

# Build a pipeline — scaler + model in one clean object
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42, max_iter=200))
])

# 5-fold cross-validation
lr_scores = cross_val_score(lr_pipeline, X, y, cv=5, scoring='accuracy')

print("LR Scores per fold:", lr_scores.round(4))
print(f"LR Mean Accuracy  : {lr_scores.mean():.4f}")
print(f"LR Std Deviation  : {lr_scores.std():.4f}")

LR Scores per fold: [0.8045 0.8034 0.8258 0.809  0.8933]
LR Mean Accuracy  : 0.8272
LR Std Deviation  : 0.0340


In [3]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ('model', RandomForestClassifier(
        n_estimators=100,    # 100 trees
        random_state=42,
        max_depth=6,         # prevent overfitting
        min_samples_split=4  # min samples to split a node
    ))
])

# Note: Random Forest doesn't need scaling — trees are scale-invariant
rf_scores = cross_val_score(rf_pipeline, X, y, cv=5, scoring='accuracy')

print("RF Scores per fold:", rf_scores.round(4))
print(f"RF Mean Accuracy  : {rf_scores.mean():.4f}")
print(f"RF Std Deviation  : {rf_scores.std():.4f}")

RF Scores per fold: [0.8436 0.8315 0.8483 0.7978 0.8652]
RF Mean Accuracy  : 0.8373
RF Std Deviation  : 0.0225


In [4]:
print("\n====== Model Comparison ======")
print(f"Logistic Regression : {lr_scores.mean():.4f} ± {lr_scores.std():.4f}")
print(f"Random Forest       : {rf_scores.mean():.4f} ± {rf_scores.std():.4f}")

if rf_scores.mean() > lr_scores.mean():
    print("\nRandom Forest wins → use it for submission")
else:
    print("\nLogistic Regression holds → simpler model is competitive")


====== Model Comparison ======
Logistic Regression : 0.8272 ± 0.0340
Random Forest       : 0.8373 ± 0.0225

Random Forest wins → use it for submission


In [5]:
# Train RF on full data to get importances
rf_model = RandomForestClassifier(
    n_estimators=100, random_state=42,
    max_depth=6, min_samples_split=4
)
rf_model.fit(X, y)

importance_df = pd.DataFrame({
    'Feature'   : X.columns,
    'Importance': rf_model.feature_importances_.round(4)
}).sort_values('Importance', ascending=False)

print("\n=== Random Forest Feature Importance ===")
print(importance_df.head(10).to_string(index=False))


=== Random Forest Feature Importance ===
           Feature  Importance
               Sex      0.1797
          Title_Mr      0.1661
              Fare      0.0894
  Sex_Class_male_3      0.0703
               Age      0.0553
        FamilySize      0.0525
            Pclass      0.0519
        Title_Miss      0.0487
Sex_Class_female_2      0.0392
         Title_Mrs      0.0351


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import pandas as pd

# X and y are your full training features from Day 4
# test is your cleaned + engineered test set

# Final Random Forest — trained on ALL training data
final_model = RandomForestClassifier(
    n_estimators=200,      # more trees = more stable
    random_state=42,
    max_depth=6,
    min_samples_split=4
)

final_model.fit(X, y)
print("Final model trained on full data ✅")
print(f"Training rows used: {X.shape[0]}")

Final model trained on full data ✅
Training rows used: 891


In [7]:
# Predict survival for test passengers
test_preds = final_model.predict(test)

print(f"Predicted Survived : {test_preds.sum()}")
print(f"Predicted Died     : {(test_preds == 0).sum()}")
print(f"Survival Rate      : {test_preds.mean():.2%}")

Predicted Survived : 151
Predicted Died     : 267
Survival Rate      : 36.12%


In [8]:
test_raw = pd.read_csv('/kaggle/input/datasets/faisalimam19/titanicdata/test.csv')

submission = pd.DataFrame({
    'PassengerId': test_raw['PassengerId'],
    'Survived'   : test_preds
})

submission.to_csv('submission.csv', index=False)

print(submission.head(10))
print(f"\nSubmission shape: {submission.shape}")
print("submission.csv saved ✅")

   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0

Submission shape: (418, 2)
submission.csv saved ✅
